# Step 7b: fixing the AdaBN arm (and re-checking the shift correlations)

**What went wrong in Step 7.** The `adabn` arm re-estimated BatchNorm statistics by passing the fine-tuning data through the model **in stored order**. The stored order is grouped by class (my stratified split concatenates class blocks), so each batch contained one or two classes, and the averaged within-batch variances omit the between-batch variance. That underestimates the variance and damages accuracy (Blood ResNet accuracy fell from 0.947 to 0.877). **All Step 7 `adabn` results (H3) are therefore invalid.** The gradient fine-tuning arms shuffle every epoch and are not affected.

**What this notebook does.** It retrains the base models with the same seeds and splits, then runs three AdaBN variants:
- `seen:adabn` and `fresh:adabn`: batches in random order (the correct procedure; batches under 32 examples are skipped).
- `seen:adabn_sorted`: the old class-sorted order, kept as a diagnostic. If the bug explanation is right, it should reproduce the large gaps and accuracy loss from Step 7, and the shuffled versions should not.

It then reloads your saved Step 7 results (`OLD_DIR`), drops the old `adabn` rows, adds the new ones, and recomputes the Spearman table. Base models retrained on a GPU can differ slightly from the Step 7 base models (non-deterministic kernels), so paired contrasts across Step 7 and 7b arms are approximate. Expected time: about 20 minutes on a GPU (not measured).

**Hypothesis (H3, restated).** Re-estimating BatchNorm statistics on in-distribution data without any gradient step changes the reuse coverage gap. No prediction is made about its size or sign.

In [ ]:
!pip install -q medmnist scipy
import os, sys, copy, math, time, glob, itertools
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- CONFIG ----
SMOKE = False                      # True = synthetic data, tests the code only (never a result)
DATASETS = ["dermamnist", "bloodmnist"]   # drop names to run fewer
SEEDS = list(range(10))
EPOCHS = {"pathmnist": 8, "dermamnist": 15, "bloodmnist": 12}
FT_EPOCHS = 2                      # default fine-tune epochs
INTENSITY = [0.5, 1, 4]              # extra fine-tune lengths (epochs) for ft_full and ft_bnfrozen on seen data
ARCHS = ["resnet", "plain"]
N_SPLITS = 40                      # random calibration/eval splits per seed
ALPHA = 0.1
USE_DRIVE = False                  # True = save results to Drive so a runtime restart does not lose finished datasets
OUT_DIR = "/content/step7b_out"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/step7b_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("device:", DEVICE, "| smoke:", SMOKE, "| datasets:", DATASETS, "| seeds:", SEEDS, "| out:", OUT_DIR)
if DEVICE == "cpu": print("WARNING: CPU only. Switch to GPU (Runtime > Change runtime type) or run one dataset and one seed.")

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
OLD_DIR = "/content/drive/MyDrive/step7_out" if USE_DRIVE else "/content/step7_out"   # where the Step 7 CSV files are
print("old Step 7 results expected in:", OLD_DIR, "| found:", os.path.isdir(OLD_DIR))


## Code

In [ ]:
# %%
def lac_all(probs):
    """Score of every label: 1 - p(label). probs: (n, K)."""
    return 1.0 - probs

def aps_all(probs):
    """Deterministic APS score of every label: total prob mass of labels ranked at or above it."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    return np.take_along_axis(cum, ranks, axis=1)

def raps_all(probs, lam=0.05, k_reg=2):
    """Deterministic RAPS-style score: APS mass plus lam * max(0, rank - k_reg). Fixed hyperparameters, not tuned."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    pen = lam * np.maximum(0, np.arange(1, probs.shape[1] + 1)[None, :] - k_reg)
    return np.take_along_axis(cum + pen, ranks, axis=1)

SCORES = {"LAC": lac_all, "APS": aps_all, "RAPS": raps_all}

def conformal_q(true_scores, alpha=ALPHA):
    """Finite-sample split-conformal quantile. Returns inf if the calibration set is too small."""
    n = len(true_scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n:
        return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=10):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan)


In [ ]:
# %%
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x))); o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))

class ResNet18Small(nn.Module):
    """ResNet-18 layout (2,2,2,2 blocks) with a 3x3 stem, suited to 28x28 images."""
    def __init__(self, num_classes, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())
        cfg, layers, cin = [(w, 1), (w, 1), (2*w, 2), (2*w, 1), (4*w, 2), (4*w, 1), (8*w, 2), (8*w, 1)], [], w
        for cout, s in cfg:
            layers.append(Block(cin, cout, s)); cin = cout
        self.layers = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def train_model(model, X, y, epochs, lr=2e-3, bs=256, seed=0):
    set_seed(seed); model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def predict_probs(model, X, bs=512):
    model.to(DEVICE).eval(); out = []
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs):
        out.append(F.softmax(model(Xt[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy().astype(np.float64)

class PlainCNN(nn.Module):
    """VGG-style CNN, no skip connections: conv-BN-ReLU x6 with two max-pools, then global average pool and a linear layer."""
    def __init__(self, num_classes, width=64):
        super().__init__(); w = width
        cb = lambda i, o: [nn.Conv2d(i, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU()]
        layers = cb(3, w) + cb(w, w) + [nn.MaxPool2d(2)] + cb(w, 2*w) + cb(2*w, 2*w) + [nn.MaxPool2d(2)] + cb(2*w, 4*w) + cb(4*w, 4*w)
        self.body = nn.Sequential(*layers); self.fc = nn.Linear(4*w, num_classes)
    def forward(self, x):
        return self.fc(F.adaptive_avg_pool2d(self.body(x), 1).flatten(1))

ARCH = "resnet"
def build_model(arch, K):
    w = 16 if SMOKE else 64
    return ResNet18Small(K, width=w) if arch == "resnet" else PlainCNN(K, width=w)


def bn_modules(m):
    return [x for x in m.modules() if isinstance(x, nn.modules.batchnorm._BatchNorm)]

def adapt(model, X, y, epochs, bn_stats="update", bn_affine="train", lr=5e-4, bs=256, seed=0):
    """Fine-tune conv/linear weights. bn_stats: 'update' (BN in train mode) or 'frozen' (BN in eval mode).
    bn_affine: 'train' or 'frozen' (BN scale and shift). One-cycle schedule over round(epochs * steps_per_epoch) steps."""
    m = copy.deepcopy(model); set_seed(seed + 99); m.to(DEVICE).train()
    bns = bn_modules(m)
    if bn_stats == "frozen":
        for b in bns: b.eval()
    if bn_affine == "frozen":
        for b in bns:
            for p in b.parameters(): p.requires_grad_(False)
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    total = max(1, int(round(epochs * math.ceil(len(y) / bs))))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=total)
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long(); steps = 0
    while steps < total:
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            if steps >= total: break
            idx = perm[i:i + bs]
            loss = F.cross_entropy(m(Xt[idx].to(DEVICE)), yt[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); steps += 1
    for p in m.parameters(): p.requires_grad_(True)
    return m.eval()

@torch.no_grad()
def adabn(model, X, bs=256):
    """Label-free: re-estimate BN running statistics on X (cumulative average over one pass); no weight update."""
    m = copy.deepcopy(model).to(DEVICE).train()
    for b in bn_modules(m): b.reset_running_stats(); b.momentum = None
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs): m(Xt[i:i + bs].to(DEVICE))
    for b in bn_modules(m): b.momentum = 0.1
    return m.eval()

def strat_split(y, frac, rng):
    a, b = [], []
    for c in np.unique(y):
        idx = rng.permutation(np.where(y == c)[0]); k = int(round(len(idx) * frac))
        if len(idx) >= 2: k = min(max(k, 1), len(idx) - 1)
        a.append(idx[:k]); b.append(idx[k:])
    return np.concatenate(a), np.concatenate(b)

def conformal_q(true_scores, alpha=ALPHA):
    n = len(true_scores); k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n: return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=5):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan,
                cls_dev=float(np.mean(np.abs(np.array(cls) - (1 - ALPHA)))) if cls else np.nan)

def mean_size(S, q):
    return float((S <= q).sum(1).mean())

def size_match_q(S_c, target_size, iters=40):
    """Label-free: smallest threshold whose mean set size on unlabeled inputs reaches target_size (bisection)."""
    lo, hi = 0.0, float(S_c.max()) + 1e-9
    for _ in range(iters):
        mid = (lo + hi) / 2
        if mean_size(S_c, mid) < target_size: lo = mid
        else: hi = mid
    return hi

def make_arms(base, XA, yA, XB, yB, XB1, yB1, seed):
    """name -> (model, group). Models are built lazily in order; all start from the same base model."""
    arms = {}
    for d, (X, y) in {"seen": (XA, yA), "fresh": (XB, yB)}.items():
        arms[f"{d}:ft_full@2"]        = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "update", "train", seed=seed)
        arms[f"{d}:ft_statsfrozen@2"] = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "frozen", "train", seed=seed)
        arms[f"{d}:ft_affinefrozen@2"] = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "update", "frozen", seed=seed)
        arms[f"{d}:ft_bnfrozen@2"]    = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "frozen", "frozen", seed=seed)
        arms[f"{d}:adabn"]            = lambda X=X: adabn(base, X)
    for e in INTENSITY:
        arms[f"seen:ft_full@{e}"]     = lambda e=e: adapt(base, XA, yA, e, "update", "train", seed=seed)
        arms[f"seen:ft_bnfrozen@{e}"] = lambda e=e: adapt(base, XA, yA, e, "frozen", "frozen", seed=seed)
    arms["fresh_half:ft_full@2"]      = lambda: adapt(base, XB1, yB1, FT_EPOCHS, "update", "train", seed=seed)
    return arms

def shift_stats(P0, P1, y, q0):
    """Score-shift diagnostics on the labeled pool (APS). P0 base probs, P1 adapted probs, q0 base threshold."""
    from scipy.stats import ks_2samp
    S0, S1 = aps_all(P0), aps_all(P1); n = len(y); ar = np.arange(n)
    s0, s1 = S0[ar, y], S1[ar, y]; D = np.abs(s1 - s0)
    ent = lambda P: float(np.mean(-(P * np.log(P + 1e-12)).sum(1)))
    r0 = (P0 > P0[ar, y][:, None]).sum(1); r1 = (P1 > P1[ar, y][:, None]).sum(1)
    return dict(eps50=float(np.quantile(D, .5)), eps90=float(np.quantile(D, .9)), eps99=float(np.quantile(D, .99)),
                mean_shift=float(np.mean(s1 - s0)), ks=float(ks_2samp(s0, s1).statistic),
                cdf_id=float(np.mean(s1 <= q0) - np.mean(s0 <= q0)),
                d_acc=float(np.mean(P1.argmax(1) == y) - np.mean(P0.argmax(1) == y)),
                d_ptrue=float(np.mean(P1[ar, y]) - np.mean(P0[ar, y])),
                d_maxp=float(P1.max(1).mean() - P0.max(1).mean()), d_ent=ent(P1) - ent(P0),
                d_size=mean_size(S1, q0) - mean_size(S0, q0),
                frac_rank_changed=float(np.mean(r0 != r1)), frac_top1_changed=float(np.mean(P0.argmax(1) != P1.argmax(1))))


@torch.no_grad()
def adabn(model, X, bs=256, shuffle=True, seed=0, min_batch=32):
    """Label-free: re-estimate BN running statistics on X (cumulative average over one pass, no weight update).
    shuffle=True draws batches in random order (correct); shuffle=False uses the stored order (the Step 7 bug if X is class-sorted)."""
    m = copy.deepcopy(model).to(DEVICE).train()
    for b in bn_modules(m): b.reset_running_stats(); b.momentum = None
    idx = np.random.RandomState(seed).permutation(len(X)) if shuffle else np.arange(len(X))
    Xt = torch.as_tensor(X)
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        if len(j) >= min_batch: m(Xt[j].to(DEVICE))
    for b in bn_modules(m): b.momentum = 0.1
    return m.eval()


## Data and experiment

In [ ]:
def load_medmnist(name, smoke):
    if smoke:
        rng = np.random.RandomState(0); K = 7
        def make(n):
            y = rng.randint(0, K, n); X = rng.rand(n, 3, 28, 28).astype(np.float32) * 0.8
            for k in range(K):
                r, c, ch = (k // 3) * 9 + 2, (k % 3) * 9 + 2, k % 3
                X[y == k, ch, r:r + 8, c:c + 8] += 0.6
            return (X - 0.5) / 0.5, y
        Xtr, ytr = make(2500); Xev, yev = make(1500)
        return Xtr, ytr, Xev, yev, K
    import medmnist
    from medmnist import INFO
    cls = getattr(medmnist, INFO[name]["python_class"])
    f = lambda a: ((a.imgs.astype(np.float32) / 255.0 - 0.5) / 0.5).transpose(0, 3, 1, 2).copy()
    lab = lambda a: a.labels.squeeze().astype(np.int64)
    tr, va, te = (cls(split=s, download=True) for s in ("train", "val", "test"))
    K = len(INFO[name]["label"])
    Xev, yev = np.concatenate([f(va), f(te)]), np.concatenate([lab(va), lab(te)])
    return f(tr), lab(tr), Xev, yev, K

def run_seed(name, arch, seed, Xtr, ytr, Xev, yev, K):
    t0 = time.time(); epochs = 3 if SMOKE else EPOCHS[name]
    rng = np.random.RandomState(777 + seed)
    iA, iB = strat_split(ytr, 0.5, rng)
    XA, yA, XB, yB = Xtr[iA], ytr[iA], Xtr[iB], ytr[iB]
    base = train_model(build_model(arch, K), XA, yA, epochs, seed=seed)
    P0 = predict_probs(base, Xev); probs = {"base": P0}
    probs["seen:adabn"] = predict_probs(adabn(base, XA, shuffle=True, seed=seed), Xev)
    probs["fresh:adabn"] = predict_probs(adabn(base, XB, shuffle=True, seed=seed), Xev)
    probs["seen:adabn_sorted"] = predict_probs(adabn(base, XA, shuffle=False), Xev)
    accs = {a: float(np.mean(p.argmax(1) == yev)) for a, p in probs.items()}
    rows, srows = [], []
    for sname, sfn in SCORES.items():
        S = {a: sfn(p) for a, p in probs.items()}
        for sp in range(N_SPLITS):
            r_ = np.random.RandomState(1000 * seed + sp)
            perm = r_.permutation(len(yev)); cal, ev = perm[:len(yev) // 2], perm[len(yev) // 2:]
            q0 = conformal_q(S["base"][cal, yev[cal]]); target = mean_size(S["base"][cal], q0)
            for a in probs:
                qs = [("reuse", q0), ("sizematch", size_match_q(S[a][cal], target)), ("recal", conformal_q(S[a][cal, yev[cal]]))]
                for mode, q in qs:
                    rows.append(dict(dataset=name, arch=arch, seed=seed, split=sp, score=sname, arm=a, mode=mode, acc=accs[a],
                                     n_cal=len(cal), **summarize(S[a][ev] <= q, yev[ev])))
    q_pool = conformal_q(aps_all(P0)[np.arange(len(yev)), yev])
    for a in probs:
        if a != "base": srows.append(dict(dataset=name, arch=arch, seed=seed, arm=a, **shift_stats(P0, probs[a], yev, q_pool)))
    print(f"  [{name}|{arch}] seed {seed} done in {time.time()-t0:.0f}s | acc " + " ".join(f"{a}={v:.3f}" for a, v in accs.items()), flush=True)
    return pd.DataFrame(rows), pd.DataFrame(srows)

def run_cell(name, arch):
    Xtr, ytr, Xev, yev, K = load_medmnist(name, SMOKE)
    print(f"[{name} | {arch}] train {len(ytr)} -> A {len(ytr)//2} / B {len(ytr)-len(ytr)//2} | pool {len(yev)} | classes {K}")
    R, Sx = [], []
    for seed in SEEDS:
        f1 = os.path.join(OUT_DIR, f"{name}_{arch}_s{seed}.csv"); f2 = os.path.join(OUT_DIR, f"{name}_{arch}_s{seed}_shift.csv")
        if os.path.exists(f1) and os.path.exists(f2) and not SMOKE:
            print(f"  seed {seed}: found saved results, loading"); r, s = pd.read_csv(f1), pd.read_csv(f2)
        else:
            r, s = run_seed(name, arch, seed, Xtr, ytr, Xev, yev, K)
            if not SMOKE: r.to_csv(f1, index=False); s.to_csv(f2, index=False)
        R.append(r); Sx.append(s)
    return pd.concat(R), pd.concat(Sx)


## Reporting

In [ ]:
from scipy import stats
def ci95(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; n = len(x)
    return float("nan") if n < 2 else stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
def pm(x): return "{:+.2f} +/- {:.2f}".format(np.nanmean(x), ci95(x))

def report_cell(df, shift):
    if SMOKE: print("*** SMOKE = True: SYNTHETIC DATA, NOT A RESULT ***")
    name, arch = df["dataset"].iloc[0], df["arch"].iloc[0]; tag = f"{name} | {arch}"
    p = df.groupby(["seed", "score", "arm", "mode"])[["coverage", "size", "worst_class_cov", "acc"]].mean().reset_index()
    p["gap"] = 100 * (p["coverage"] - (1 - ALPHA)); ns = p["seed"].nunique()
    print(f"\n################ {tag}  ({ns} seeds) ################")
    print("\nTop-1 accuracy on the pool (mean over seeds):")
    print(p[(p["mode"] == "reuse") & (p["score"] == "APS")].groupby("arm")["acc"].mean().round(4).to_string())
    for sc in ["APS", "LAC", "RAPS"]:
        print(f"\n===== {tag} | {sc}: coverage gap in points (negative = undercoverage), mean +/- 95% t over {ns} seeds =====")
        rows = []
        for a in [x for x in p["arm"].unique() if x != "base"]:
            row = dict(arm=a)
            for m in ["reuse", "sizematch", "recal"]:
                row[m] = pm(p[(p["score"] == sc) & (p["arm"] == a) & (p["mode"] == m)]["gap"])
            rows.append(row)
        print(pd.DataFrame(rows).to_string(index=False))
    print(f"\n===== {tag} | SCORE-SHIFT DIAGNOSTICS (APS, mean over seeds) =====")
    print(shift.groupby("arm")[["eps50", "eps90", "eps99", "ks", "mean_shift", "d_acc", "d_maxp", "d_ent", "d_size", "frac_top1_changed"]].mean().round(4).to_string())
    return p

def merged_correlation(ps, shifts):
    """Spearman of shift diagnostics vs APS reuse gap, using the saved Step 7 arms (old adabn dropped) plus the new adabn arms."""
    rows = []
    for k in ps:
        name, arch = k.split("|")
        old_r, old_s = [], []
        for seed in SEEDS:
            f1 = os.path.join(OLD_DIR, f"{name}_{arch}_s{seed}.csv"); f2 = os.path.join(OLD_DIR, f"{name}_{arch}_s{seed}_shift.csv")
            if os.path.exists(f1) and os.path.exists(f2): old_r.append(pd.read_csv(f1)); old_s.append(pd.read_csv(f2))
        if not old_r:
            print(f"  [{k}] no saved Step 7 files in {OLD_DIR}; skipping merge for this cell"); continue
        o_r, o_s = pd.concat(old_r), pd.concat(old_s)
        o_r = o_r[~o_r["arm"].str.contains("adabn")]; o_s = o_s[~o_s["arm"].str.contains("adabn")]
        for tagname, r_, s_ in [("old arms", o_r, o_s), ("new adabn arms", None, shifts[k])]:
            pass
        g_old = o_r[(o_r["score"] == "APS") & (o_r["mode"] == "reuse")].groupby(["seed", "arm"])["coverage"].mean().reset_index()
        g_new = ps[k][(ps[k]["score"] == "APS") & (ps[k]["mode"] == "reuse") & (ps[k]["arm"] != "base")][["seed", "arm", "coverage"]]
        gp = pd.concat([g_old, g_new]); gp["gap"] = 100 * (gp["coverage"] - (1 - ALPHA))
        sh = pd.concat([o_s, shifts[k]])
        j = sh.merge(gp[["seed", "arm", "gap"]], on=["seed", "arm"]); j["cell"] = k; rows.append(j)
    if not rows: return
    J = pd.concat(rows)
    for title, jj in [("ALL ARMS (old Step 7 arms + corrected adabn arms; adabn_sorted excluded)", J[~J["arm"].str.contains("sorted")]),
                      ("EXCLUDING ALL adabn ARMS", J[~J["arm"].str.contains("adabn")])]:
        print(f"\n################ SPEARMAN of shift diagnostics with the APS reuse gap: {title}; n = {len(jj)} ################")
        out = []
        for c in ["eps50", "eps90", "eps99", "ks", "mean_shift", "d_acc", "d_ptrue", "d_maxp", "d_ent", "d_size", "frac_rank_changed", "frac_top1_changed"]:
            r, _ = stats.spearmanr(jj[c], jj["gap"]); out.append(dict(stat=c, spearman_rho=round(r, 3), label_free=c in ("d_maxp", "d_ent", "d_size", "frac_top1_changed")))
        print(pd.DataFrame(out).to_string(index=False))
        print("ks is unsigned, so compare it with |gap|:", round(stats.spearmanr(jj["ks"], jj["gap"].abs())[0], 3))
    print("\nCaution: arms within a seed share a base model, so rows are not independent; rho is descriptive. d_size is mechanically linked to coverage (both are functionals of the score distribution at the old threshold), so it is a diagnostic, not independent evidence of a mechanism.")


## Run

In [ ]:
ps, shifts = {}, {}
for arch in ARCHS:
    for name in DATASETS:
        df, sh = run_cell(name, arch)
        ps[f"{name}|{arch}"] = report_cell(df, sh); shifts[f"{name}|{arch}"] = sh
merged_correlation(ps, shifts)
print("\nRead: (1) does adabn_sorted reproduce the Step 7 gaps and accuracy loss while shuffled adabn does not (confirms the bug)? (2) what does correct AdaBN do to the gap and to accuracy? (3) how do the correlations change without the contaminated rows?")